### ETL: bronze.gastronomy -> silver.visit_points

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType
from delta.tables import DeltaTable 
import sys
import os
from pathlib import Path
current_dir = "/Workspace" + os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
src_path = str(Path(current_dir).parents[1])
sys.path.append(src_path)

from utils.cleaning_functions import *

In [0]:
df_source = spark.table("dbw_routemind_euskadi_dev.bronze.gastronomy")

In [0]:
cast_map = {
    "documentName": T.StringType(),
    "templateType": T.StringType(),
    "restorationType": T.StringType(),
    "latwgs84": T.DoubleType(),
    "lonwgs84": T.DoubleType(),
    "municipalitycode": T.StringType(),
    "territorycode": T.StringType(),
}

notnull_columns = [
    "documentName",
    "templateType",
    "municipalitycode",
    "territorycode"
]

keys=["documentName", "category"]

In [0]:
df_exploded = explode_array_column(df_source,"rows.row","row_item")

In [0]:
df_filtered = df_exploded.filter(F.col("row_item.restorationType") =! "Restaurante")


In [0]:

df_extracted = extract_and_cast(df_filtered,"row_item",cast_map)

In [0]:
df_clean = drop_null_required(df_extracted, notnull_columns)


In [0]:
df_final = add_category_column(df_clean, category_value="gastronomy")

In [0]:
df_final = deduplicate(df_final, keys)

In [0]:
df_sorted = df_final.withColumnRenamed("restorationType","marks")

In [0]:
target_table = "dbw_routemind_euskadi_dev.silver.visit_points"

delta_target = DeltaTable.forName(spark, target_table)
(dbw_routemind_euskadi_dev.silver.visit_points
    delta_target.alias("t")
    .merge(
        df_final.alias("s"),
        "t.documentName = s.documentName AND t.category = s.category"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
print(f"MERGE completed on {target_table}. rows processed: {df_final.count()}")